In [463]:
import ast
import json
import pandas as pd

## Data Cleaning

In [464]:
df1 = pd.read_csv('data/Annotated Data/o4mini_annotation 1(in).csv')[['vaers_id', 'llm_result']]
df2 = pd.read_excel('data/Annotated Data/o4mini_annotation_verified.xlsx', sheet_name='data')
df2.drop(columns=['llm_result'], inplace=True, axis=1)
data = df1.merge(df2, on='vaers_id', how='left')

In [465]:
data.rename(
    columns={
        'Correct/Incorrect/Uncertain': 't_result',
        'corrected_version': 't_corrected_version',
        'category': 't_category',
        'comment': 't_notes',
        'H_Result(0: Incorrect, 1: Correct, 2: Uncertain)': 'h_result',
        'final_results': 'final_result'
    }, inplace=True)

In [466]:
data.columns

Index(['vaers_id', 'llm_result', 'symptom_text', 'symptom_list', 't_result',
       't_corrected_version', 't_category', 't_notes', 'h_result',
       'h_corrected_version', 'h_notes', 'final_result',
       'final_corrected_version', 'final_notes', 'num_groups',
       'symptom_match_flag', 'llm_result_cleaned'],
      dtype='object')

In [467]:
data = data[[
    'vaers_id', 'llm_result', 'symptom_text', 'symptom_list', 't_result',
    't_corrected_version', 't_category', 't_notes', 'h_result',
    'h_corrected_version', 'h_notes', 'final_result',
    'final_corrected_version', 'final_notes'
]]

In [468]:
data.head()

,vaers_id,llm_result,symptom_text,symptom_list,t_result,t_corrected_version,t_category,t_notes,h_result,h_corrected_version,h_notes,final_result,final_corrected_version,final_notes
0,1420769,"[ \r\n { \r\n ""Chest pain"": [""non stop c...","About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...",correct,NaN,NaN,NaN,1,NaN,NaN,1.0,NaN,NaN
1,1527339,"""No temporal information found""","cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...",correct,NaN,NaN,NaN,1,NaN,NaN,1.0,NaN,NaN
2,1711389,"[\r\n {\r\n ""Lymphadenopathy"": [""lymph nod...",right side lymph swelling; right ear pain; cou...,"['Ear pain', 'Headache', 'Hypoacusis', 'Lympha...",incorrect,"[\n {\n ""Lymphadenopathy"": [\n ""lymph...",Symptoms with no original mention should be gr...,NaN,0,"[\n {\n ""Lymphadenopathy"": [""lymph node sw...",format,NaN,NaN,NaN
3,1021935,"[\r\n {\r\n ""Vaccination site pain"": [""inj...",back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...",correct,NaN,NaN,NaN,0,"[\n {\n ""Vaccination site pain"": [""injecti...",Wrong order.,1.0,NaN,NaN
4,2749020,"[\r\n {\r\n ""Syncope"": [""patient fainted""]...",The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...",correct,NaN,NaN,NaN,1,NaN,NaN,1.0,NaN,NaN


In [469]:
def clean_and_parse_json(text):
    if isinstance(text, str):
        try:
            text = text.strip().removeprefix('"').removesuffix('"')
            if text == 'No temporal information found':
                return text

            if text.startswith('```json') and text.endswith('```'):
                text = text.removeprefix("```json").removesuffix("```").strip()

            return json.loads(text)

        except json.JSONDecodeError:
            print('Error decoding JSON', type(text), text)
            return 'json_error'

    return None

def verify_symptoms(row):
    symptom_list = row.symptom_list
    llm_output = row.llm_result_cleaned
    if not isinstance(llm_output, list):
        return None

    key_list = [key for d in llm_output if isinstance(d, dict) for key in d.keys()]
    return set(symptom_list) == set(key_list)

def find_missing_symptoms(row):
    if row.all_symptoms_extracted == False:
        original_symptoms = set(row.symptom_list)
        extracted_symptoms = set(
            key for d in row.llm_result_cleaned if isinstance(d, dict) for key in d.keys()
        )
        missing_symptoms = {
            'original_symptoms': list(original_symptoms.difference(extracted_symptoms)),
            'extracted_symptoms': list(extracted_symptoms.difference(original_symptoms))
        }
        return json.dumps(missing_symptoms, indent=2)

    return None

def find_num_groups(row):
    if isinstance(row.llm_result_cleaned, list):
        return len(row.llm_result_cleaned)

    return None

def filter_symptom_none(row):

    try:
        if isinstance(row.llm_result_cleaned, list):
            original_symptoms_set = set(row.symptom_list)
            llm_result = row.llm_result_cleaned

            cleaned_annotated_list = []
            syms_with_original_mention = set()
            none_values = ['none', ['none']]
            generic_terms = ['Condition aggravated']

            for syms_grp in llm_result:
                new_syms_grp = {k: v for k, v in syms_grp.items() if not (k in generic_terms or v in none_values)}
                syms_with_original_mention.update(new_syms_grp.keys())
                if new_syms_grp:
                    cleaned_annotated_list.append(new_syms_grp)

            syms_without_original_mention = original_symptoms_set.difference(syms_with_original_mention)
            if syms_without_original_mention:
                cleaned_annotated_list.append(
                    {symptom : ['none'] for symptom in syms_without_original_mention}
                )

            return json.dumps(cleaned_annotated_list, indent=2)

    except Exception:
        print(row.vaers_id, row.llm_result_cleaned)
        return row.llm_result_cleaned

    return row.llm_result_cleaned

def llm_errors(row):
    error = ''
    llm_result = row.llm_result.strip()

    if llm_result.startswith('```json') and llm_result.endswith('```'):
        error += 'Formatting error (```json```)\n'

    if row.all_symptoms_extracted == False:
        error += 'Missing symptoms\n'

    return error

def calculate_agreement(row):
    return row.t_result == row.h_result == 1 and row.final_result == 1

In [470]:
data['symptom_list'] = data['symptom_list'].apply(ast.literal_eval)
data['llm_result_cleaned'] = data['llm_result'].apply(clean_and_parse_json)
data['all_symptoms_extracted'] = data.apply(verify_symptoms, axis=1)
data['missing_symptoms'] = data.apply(find_missing_symptoms, axis=1)
data['num_groups'] = data.apply(find_num_groups, axis=1)
data['llm_result_formatted'] = data.apply(filter_symptom_none, axis=1)
data['t_result'] = data['t_result'].str.strip().str.lower().map(
    {'uncertain': 2, 'incorrect': 0, 'correct': 1})
data['llm_errors'] = data.apply(llm_errors, axis=1)
data['agreement'] = data.apply(calculate_agreement, axis=1)

In [471]:
df_final = data[[
    'vaers_id', 'symptom_text', 'symptom_list', 'llm_result', 'llm_result_formatted',
    't_result', 't_corrected_version', 't_category', 't_notes', 'h_result',
    'h_corrected_version', 'h_notes', 'final_result', 'final_corrected_version',
    'final_notes', 'all_symptoms_extracted', 'missing_symptoms', 'llm_errors',
    'num_groups', 'agreement'
]]

In [472]:
filename = 'data/Annotated Data/o4mini_annotation_verified.xlsx'
with pd.ExcelWriter(filename, mode='a') as writer:
    df_final.to_excel(writer, sheet_name='data2', index=False)